<a href="https://colab.research.google.com/github/Manjit-yadav/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Manjit-yadav/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My lane is a **classification problem**. The goal is to classify each content item into a high-priority or lower-priority content refresh opportunity.

The model would use available content and search characteristics such as search volume, competition, content type, search intent, word count, and other relevant attributes to identify content that may need attention. The output would support a content team in deciding which existing pages should be reviewed or refreshed first.

Classification is appropriate because the final decision can be represented as categories such as **high-priority refresh** and **not high-priority refresh**, rather than only producing a continuous number.


In [2]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7



## 2. Target or proxy

My target is a binary **refresh-priority proxy**.

I use observed CTR (`ctr`) to define the label. Content with CTR below the dataset median is labeled `1`, meaning it is a potential refresh candidate, while content at or above the median is labeled `0`.

The target therefore comes from an observed performance measure rather than from a manually invented category. The median threshold is only a starting rule for creating a proxy label; it does not mean that every low-CTR page necessarily needs to be refreshed.

The target would look like:

* `1` = potential refresh candidate based on lower-than-median CTR
* `0` = not prioritized by this proxy


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["ctr"].describe()

,ctr
count,30000.000000
mean,0.510733
std,3.279162
min,0.000000
25%,0.000000
50%,0.070000
75%,0.290000
max,100.000000


In [4]:
ctr_threshold = df["ctr"].median()

df["refresh_priority"] = (
    df["ctr"] < ctr_threshold
).astype(int)

df[["content_id", "ctr", "refresh_priority"]].head(10)

,content_id,ctr,refresh_priority
0,content_304f48230142,0.76,0
1,content_a1fb4e703a9e,0.05,1
2,content_9aa793d4d895,0.09,0
3,content_331d6c4de07b,0.49,0
4,content_d99b7a2d90ca,0.13,0
5,content_d4084a4bc775,0.03,1
6,content_9a34b442b552,0.00,1
7,content_a63219c6e95a,0.06,1
8,content_5e6c160719bc,0.09,0
9,content_c27558df2b0c,0.16,0


## 3. Success metric

I would use **F1-score** as the primary success metric.

F1-score balances precision and recall. This matters because the goal is to identify content that may need refreshing without producing too many incorrect refresh recommendations.

A higher F1-score means the classifier is doing a better job of identifying the target class while maintaining a reasonable balance between missed candidates and false recommendations.

For this task, I would consider the model useful if it performs substantially better than a simple baseline classifier on unseen data.


## 4. The unit of analysis, as a real dataframe

**One row = one content item/page.**

Each row represents an individual piece of content associated with a client. The columns describe characteristics of that content, its search environment, and its observed performance.

For this ML task, the model would make one classification for each content item: whether it should be considered a potential high-priority refresh candidate according to the target definition.

The dataframe above shows the actual unit of analysis rather than only describing it conceptually.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
unit_df = df[
    [
        "content_id",
        "client_id",
        "search_volume",
        "competition",
        "competition_level",
        "cpc",
        "content_type",
        "main_intent",
        "word_count",
        "ctr",
        "avg_position",
        "trend_direction",
        "trend_pct"
    ]
].copy()

unit_df.head(10)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,ctr,avg_position,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,0.76,10.6,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,0.05,20.3,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,0.09,36.5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,0.49,6.2,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,0.13,44.0,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,720.0,1.00,HIGH,1.05,keyword article,transactional,3080.0,0.03,8.5,down,-38.9
6,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,informational,3059.0,0.00,7.0,down,-92.3
7,content_a63219c6e95a,client_19581e27de,590.0,0.44,MEDIUM,0.64,keyword article,commercial,NaN,0.06,21.2,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3807.0,0.09,46.0,down,-58.8
9,content_c27558df2b0c,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,0.16,4.9,down,-29.2


In [6]:
target_df = df[
    [
        "content_id",
        "ctr",
        "refresh_priority"
    ]
].copy()

target_df.head(10)

,content_id,ctr,refresh_priority
0,content_304f48230142,0.76,0
1,content_a1fb4e703a9e,0.05,1
2,content_9aa793d4d895,0.09,0
3,content_331d6c4de07b,0.49,0
4,content_d99b7a2d90ca,0.13,0
5,content_d4084a4bc775,0.03,1
6,content_9a34b442b552,0.00,1
7,content_a63219c6e95a,0.06,1
8,content_5e6c160719bc,0.09,0
9,content_c27558df2b0c,0.16,0


## 5. Why ML beats a fixed rule here

A fixed rule could identify refresh candidates using a single threshold, such as marking every page with CTR below a chosen value for refresh. However, content performance depends on multiple factors rather than one variable.

The dataset contains search volume, competition, competition level, CPC, content type, search intent, word count, CTR, average position, engagement rate, scroll rate, and trend information. These variables can interact in ways that are difficult to capture with a small set of manually chosen if-statements.

An ML classifier can learn patterns from historical content examples and combine multiple features when making a prediction. This could help the content team prioritize which pages deserve further review instead of relying only on one fixed threshold.

The final action is therefore:

**content data → ML classification → potential refresh candidates → content team reviews and prioritizes pages for refresh.**


In [9]:
X = df[
    [
        "search_volume",
        "competition",
        "competition_level",
        "cpc",
        "content_type",
        "main_intent",
        "word_count",
        "char_count"
    ]
]

## Self-check

Before you submit, confirm each line honestly:

- [ ✔️] Every section above is filled — markdown thinking AND the code that backs it
- [ ✔️] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✔️] No client names, URLs, or private queries anywhere
- [✔️ ] My claims use careful words: observed, measured, directional, decision-support
- [✔️ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.